# modeling_v11 — v10 − 시간/레짐 7개 (combined + Lot TE, 시간피처 제외)

> **modeling_v10 에서 시간/레짐 피처 7개만 제거**한 버전.
> 피처 = combined 원본센서 36 + WF집계 144 + 구조 2 + 범주형 2 + `C23_te` + `C20_te` = **186개**
> (v10 193개 − 시간/레짐 7개)
>
> 제거된 7개: `is_post_loud_pm`, `days_since_last_pm`, `hour`, `dslp_x_hour`, `hour_x_c33`, `post_pm_days`, `is_special_recipe`

**검증 결과 (클라우드 미러 재현):** 셀 5 출력 참조 (v10 = OOF 34.87 / valid 34.29 / test 34.33 대비 변화 확인용).

> ⚠️ **누수 주의** — `C20_te`(Lot 평균 C65)는 valid/test 가 train 과 Lot 을 공유해서 점수를 낮춘다.
> 신규 Lot 기준으론 무너지며 실무 일반화가 안 된다. 프로젝트 원칙(Lot 피처 금지)과 배치되는 대회 점수 전용.

**전제:** 이 노트북은 `modeling_v11/` 안에 있어야 하며, 형제 폴더
`../combined_model_full_handover/src` 와 `../문제1(하)`, `../문제1_하_answer` 를 상대경로로 참조한다. 커널: `venv`.

In [1]:
# [셀 1] 설정 — 경로, import
import sys, json
from pathlib import Path
import numpy as np, pandas as pd

CWD = Path.cwd()                                  # modeling_v11/
HANDOVER = CWD.parent / "combined_model_full_handover"
sys.path.insert(0, str(HANDOVER / "src"))

from preprocessing import preprocess
from feature_engineering import build_features
from combined_model import (build_rows, PARAMS, CAT_COLS, N_FOLDS,
                            EARLY_STOPPING, TE_COL, TIME_FEATS)
from config import ID_COL, TARGET_COL             # 'C64', 'C65'
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error

DATA = CWD.parent / "문제1(하)"
ANS  = CWD.parent / "문제1_하_answer"
PATHS = {
    "train":     DATA / "train_data.csv",
    "valid_X":   DATA / "valid_X.csv",
    "test_X":    DATA / "test_X.csv",
    "valid_ans": ANS  / "valid_Y_answer.csv",
    "test_ans":  ANS  / "test_Y_answer.csv",
}
PM_BINS = np.array(json.load(open(HANDOVER / "data/processed/pm_bins.json")))
PM_LOG  = json.load(open(HANDOVER / "data/processed/pm_log.json"))

def rmse(a, b):
    return float(np.sqrt(mean_squared_error(a, b)))

print("경로 확인:", {k: v.exists() for k, v in PATHS.items()})
print("제거 대상 시간/레짐 피처:", TIME_FEATS)

경로 확인: {'train': True, 'valid_X': True, 'test_X': True, 'valid_ans': True, 'test_ans': True}
제거 대상 시간/레짐 피처: ['is_post_loud_pm', 'days_since_last_pm', 'hour', 'dslp_x_hour', 'hour_x_c33', 'post_pm_days', 'is_special_recipe']


In [2]:
# [셀 2] combined row-level 피처 빌드 + Lot(C20) 부착
def rows_of(path, has_t):
    raw = pd.read_csv(path)
    clean = preprocess(raw.copy())
    wt, _ = build_features(clean, pm_bins=PM_BINS, pm_log=PM_LOG)
    x = raw if has_t else raw[[c for c in raw.columns if c != TARGET_COL]]
    rows, feats = build_rows(x, wt, has_target=has_t)
    lot = raw.groupby(ID_COL)["C20"].first().rename("C20")
    rows = rows.merge(lot, on=ID_COL, how="left")
    return rows, feats

rows_tr, feats = rows_of(PATHS["train"], True)
rows_va, _ = rows_of(PATHS["valid_X"], False)
rows_te, _ = rows_of(PATHS["test_X"], False)
print(f"combined 피처(시간포함) {len(feats)}개 | rows: train {len(rows_tr):,} / valid {len(rows_va):,} / test {len(rows_te):,}")

  컬럼 제거: 23개 → 잔여 42열
  시간 정렬 완료: 2018-12-01 ~ 2019-02-08
  FDC 집계 완료: 556개 피처 (수치 22개 × 5통계 × Step수 + C41_max + 문자열 1개)
  메타 파생 완료: 8개 피처
  컬럼 제거: 23개 → 잔여 41열
  시간 정렬 완료: 2018-12-01 ~ 2019-02-08
  FDC 집계 완료: 556개 피처 (수치 22개 × 5통계 × Step수 + C41_max + 문자열 1개)
  메타 파생 완료: 8개 피처
  컬럼 제거: 23개 → 잔여 41열
  시간 정렬 완료: 2018-12-01 ~ 2019-02-08
  FDC 집계 완료: 556개 피처 (수치 22개 × 5통계 × Step수 + C41_max + 문자열 1개)
  메타 파생 완료: 8개 피처
combined 피처(시간포함) 191개 | rows: train 123,614 / valid 20,577 / test 20,510


In [3]:
# [셀 3] ★ 시간/레짐 7개 제거 + C23/Lot 타깃인코딩 정의
def te_fit(frame, col, m=20):
    prior = frame[TARGET_COL].mean()
    agg = frame.groupby(col)[TARGET_COL].agg(["mean", "count"])
    enc = (agg["mean"] * agg["count"] + prior * m) / (agg["count"] + m)
    return enc, prior

for r in (rows_tr, rows_va, rows_te):
    for c in CAT_COLS:
        r[c] = r[c].astype("category")

feats_notime = [f for f in feats if f not in TIME_FEATS]     # ★ 시간/레짐 제외
USE = feats_notime + [TE_COL + "_te", "C20_te"]
print(f"제거 전 {len(feats)} → 제거 후 {len(feats_notime)} (−{len(feats)-len(feats_notime)})")
print(f"모델 입력 피처 수: {len(USE)} (combined-시간 {len(feats_notime)} + C23_te + C20_te)")

제거 전 191 → 제거 후 184 (−7)
모델 입력 피처 수: 186 (combined-시간 184 + C23_te + C20_te)


In [4]:
# [셀 4] GroupKFold(C64) 5-fold LightGBM 학습 (row 예측 → WF 평균)
y  = rows_tr[TARGET_COL].values
gr = rows_tr[ID_COL].astype("category").cat.codes.values

oof  = np.zeros(len(rows_tr))
va_p = np.zeros(len(rows_va)); te_p = np.zeros(len(rows_te))
for k, (tri, vai) in enumerate(GroupKFold(N_FOLDS).split(rows_tr, y, gr), 1):
    tf, vf = rows_tr.iloc[tri].copy(), rows_tr.iloc[vai].copy()
    vae, tee = rows_va.copy(), rows_te.copy()
    e23, p23 = te_fit(tf, TE_COL)
    for d in (tf, vf, vae, tee):
        d[TE_COL + "_te"] = d[TE_COL].map(e23).fillna(p23)
    e20, p20 = te_fit(tf, "C20")
    for d in (tf, vf, vae, tee):
        d["C20_te"] = d["C20"].map(e20).fillna(p20)
    mdl = lgb.LGBMRegressor(**PARAMS)
    mdl.fit(tf[USE], tf[TARGET_COL], eval_set=[(vf[USE], vf[TARGET_COL])],
            categorical_feature=CAT_COLS,
            callbacks=[lgb.early_stopping(EARLY_STOPPING, verbose=False),
                       lgb.log_evaluation(0)])
    oof[vai] = mdl.predict(vf[USE])
    va_p += mdl.predict(vae[USE]) / N_FOLDS
    te_p += mdl.predict(tee[USE]) / N_FOLDS
    print(f"  fold{k}/{N_FOLDS} best_iter={mdl.best_iteration_}")

  fold1/5 best_iter=283
  fold2/5 best_iter=225
  fold3/5 best_iter=217
  fold4/5 best_iter=238
  fold5/5 best_iter=213


In [5]:
# [셀 5] WF 단위 집계 + valid/test RMSE 평가
def wf(rows, p):
    return pd.DataFrame({ID_COL: rows[ID_COL].values, "p": p}).groupby(ID_COL)["p"].mean()

oof_wf = wf(rows_tr, oof); y_wf = rows_tr.groupby(ID_COL)[TARGET_COL].first()
va_wf  = wf(rows_va, va_p); te_wf = wf(rows_te, te_p)

ansv = pd.read_csv(PATHS["valid_ans"]).set_index("C64")["C65"]
anst = pd.read_csv(PATHS["test_ans"]).set_index("C64")["C65"]
iv = ansv.index.intersection(va_wf.index); it = anst.index.intersection(te_wf.index)

res = {"oof":   rmse(y_wf.loc[oof_wf.index], oof_wf),
       "valid": rmse(ansv.loc[iv], va_wf.loc[iv]),
       "test":  rmse(anst.loc[it], te_wf.loc[it])}
print(f"[modeling_v11 = v10 − 시간7] OOF={res['oof']:.4f}  valid={res['valid']:.4f}  test={res['test']:.4f}")
print(f"(참고 v10: OOF 34.87 / valid 34.29 / test 34.33)")

[modeling_v11 = v10 − 시간7] OOF=35.2191  valid=34.6897  test=35.0170
(참고 v10: OOF 34.87 / valid 34.29 / test 34.33)


In [6]:
# [셀 6] 제출 파일 + results.json 저장
out = CWD / "outputs"; out.mkdir(exist_ok=True)
va_wf.rename("predicted_C65").reset_index().rename(columns={ID_COL:"wafer_id"}).to_csv(out/"valid_Y_submit.csv", index=False)
te_wf.rename("predicted_C65").reset_index().rename(columns={ID_COL:"wafer_id"}).to_csv(out/"test_Y_submit.csv", index=False)
json.dump(res, open(out/"results.json", "w"), indent=2)
print("저장:", out/"valid_Y_submit.csv", "|", out/"test_Y_submit.csv", "|", out/"results.json")

저장: C:\Users\Dell3571\Documents\defect_test_prediction_MK\modeling_v11\outputs\valid_Y_submit.csv | C:\Users\Dell3571\Documents\defect_test_prediction_MK\modeling_v11\outputs\test_Y_submit.csv | C:\Users\Dell3571\Documents\defect_test_prediction_MK\modeling_v11\outputs\results.json
